# NBD-Dirichlet 多カテゴリ分析：K × S マッピング

複数カテゴリに対して NBD モデルとディリクレ多項分布モデルを当てはめ、  
カテゴリごとの $K$（購入頻度の異質性）と $S$（ブランドロイヤルティ）を散布図上にマッピングする。

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import gammaln
from scipy.optimize import minimize
from scipy.stats import nbinom

sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "Hiragino Sans"

## Step 1: データ読み込み

In [ ]:
DATA_DIR = "../data/dunnhumby"

df_trans = pd.read_csv(os.path.join(DATA_DIR, "transaction_data.csv"))
df_product = pd.read_csv(os.path.join(DATA_DIR, "product.csv"))

# 最初の52週間（DAY 1〜364）に絞る
min_day = df_trans["DAY"].min()
df_trans = df_trans[df_trans["DAY"] <= min_day + 7 * 52 - 1].copy()

df_trans = df_trans.merge(
    df_product[["PRODUCT_ID", "COMMODITY_DESC", "MANUFACTURER"]],
    on="PRODUCT_ID",
    how="left",
)

## Step 2: カテゴリ選定

以下の条件をすべて満たすカテゴリを分析対象とする。

- 購買世帯数 ≥ 200
- 主要メーカー（MANUFACTURER）が 3 以上
- 52週間の購買件数 ≥ 1,000

In [ ]:
df_cat_stats = (
    df_trans.groupby("COMMODITY_DESC")
    .agg(
        n_households=("household_key", "nunique"),
        n_transactions=("BASKET_ID", "count"),
        n_manufacturers=("MANUFACTURER", "nunique"),
    )
    .reset_index()
)

df_categories = df_cat_stats[
    (df_cat_stats["n_households"] >= 200)
    & (df_cat_stats["n_manufacturers"] >= 3)
    & (df_cat_stats["n_transactions"] >= 1000)
].sort_values("n_transactions", ascending=False).reset_index(drop=True)

df_categories

In [ ]:
# 分析対象カテゴリを選定（取引量上位かつメーカー数が多すぎないもの）
# メーカー数が多すぎると上位5+Othersの集約が粗くなるため、n_manufacturers <= 50 を目安にする
TARGET_CATEGORIES = [
    "SOFT DRINKS",
    "FLUID MILK PRODUCTS",
    "CHEESE",
    "BAG SNACKS",
    "COLD CEREAL",
    "YOGURT",
    "FROZEN PIZZA",
    "SOUP",
    "FRZN MEAT/MEAT DINNERS",
    "BAKED BREAD/BUNS/ROLLS",
]

print(f"分析対象カテゴリ: {len(TARGET_CATEGORIES)}")
for cat in TARGET_CATEGORIES:
    row = df_categories[df_categories["COMMODITY_DESC"] == cat]
    if len(row) > 0:
        r = row.iloc[0]
        print(f"  {cat}: {r['n_households']}世帯, {r['n_transactions']}件, {r['n_manufacturers']}メーカー")

## Step 3: カテゴリごとの NBD フィッティング

各カテゴリに対して、世帯ごとの購買回数を集計し、NBD の MLE で $(M, K)$ を推定する。

$$r_i \sim \mathrm{NegBin}(M, K), \quad \mathrm{Var}(r) = M + M^2/K$$

- $M$: 平均購買回数（プリファレンス）  
- $K$: 形状パラメータ（大きいほど消費者間の購買回数が均一）

In [ ]:
def fit_nbd(counts):
    """NBD MLE: returns (K, M) where K=shape, M=mean."""
    mean_x = counts.mean()
    var_x = counts.var()
    p_init = mean_x / var_x if var_x > mean_x else 0.5
    r_init = max(mean_x * p_init / (1 - p_init + 1e-9), 0.1)

    def neg_loglik(params):
        r, p = params
        if r <= 0 or p <= 0 or p >= 1:
            return 1e10
        return -nbinom.logpmf(counts, n=r, p=p).sum()

    result = minimize(
        neg_loglik,
        x0=[r_init, p_init],
        method="Nelder-Mead",
        options={"maxiter": 10000},
    )
    r_hat, p_hat = result.x
    K = r_hat
    M = r_hat * (1 - p_hat) / p_hat
    return K, M, result.success

In [ ]:
all_households = df_trans["household_key"].unique()

nbd_records = []
for cat in TARGET_CATEGORIES:
    df_cat = df_trans[df_trans["COMMODITY_DESC"] == cat]
    hh_counts = df_cat.groupby("household_key")["BASKET_ID"].count()
    # 購買していない世帯は0
    counts = hh_counts.reindex(all_households, fill_value=0).values

    K, M, converged = fit_nbd(counts)
    nbd_records.append({"category": cat, "M": M, "K": K, "converged": converged})
    print(f"{cat}: K={K:.3f}, M={M:.3f}, converged={converged}")

df_nbd = pd.DataFrame(nbd_records)
df_nbd

## Step 4: カテゴリごとのディリクレ多項分布フィッティング

各カテゴリについて、購買回数上位 5 MANUFACTURER + Others の 6 ブランドに集約し、  
ディリクレ多項分布の MLE で $\boldsymbol{\alpha}$ を推定、$S = \sum_j \alpha_j$ を算出する。

In [ ]:
def dm_neg_loglik(log_alpha, X, n):
    """Negative log-likelihood of Dirichlet-Multinomial distribution."""
    alpha = np.exp(log_alpha)
    S = alpha.sum()
    N = len(n)
    ll = N * gammaln(S) - gammaln(n + S).sum()
    ll += (gammaln(X + alpha) - gammaln(alpha)).sum()
    return -ll


def fit_dm(df_cat):
    """Fit DM model for a category. Returns (alpha_hat, S_hat, brand_cols, converged)."""
    mfr_counts = df_cat["MANUFACTURER"].value_counts()
    top5 = mfr_counts.head(5).index.tolist()

    df_cat = df_cat.copy()
    df_cat["brand_group"] = df_cat["MANUFACTURER"].where(
        df_cat["MANUFACTURER"].isin(top5), other="Others"
    ).astype(str)
    labels = {str(m): f"Mfr_{m}" for m in top5}
    labels["Others"] = "Others"
    df_cat["brand_group"] = df_cat["brand_group"].map(labels)

    hh_brand = (
        df_cat.groupby(["household_key", "brand_group"])["BASKET_ID"]
        .count()
        .unstack(fill_value=0)
    )
    # 購買世帯のみ（少なくとも1回購買）
    hh_brand = hh_brand[hh_brand.sum(axis=1) > 0]

    brand_cols = hh_brand.columns.tolist()
    X = hh_brand.values.astype(float)
    n = X.sum(axis=1)
    J = len(brand_cols)

    share_init = X.sum(axis=0) / X.sum()
    alpha_init = share_init * J

    result = minimize(
        dm_neg_loglik,
        x0=np.log(alpha_init),
        args=(X, n),
        method="Nelder-Mead",
        options={"maxiter": 50000, "xatol": 1e-8, "fatol": 1e-8},
    )
    alpha_hat = np.exp(result.x)
    S_hat = alpha_hat.sum()
    return alpha_hat, S_hat, brand_cols, result.success

In [ ]:
dm_records = []
for cat in TARGET_CATEGORIES:
    df_cat = df_trans[df_trans["COMMODITY_DESC"] == cat]
    _, S_hat, _, converged = fit_dm(df_cat)
    dm_records.append({"category": cat, "S": S_hat, "converged": converged})
    print(f"{cat}: S={S_hat:.3f}, converged={converged}")

df_dm = pd.DataFrame(dm_records)
df_dm

## Step 5: K × S マッピング

- x 軸: $K$（NBD 形状パラメータ）— 大きいほど消費者間の購買回数が均一
- y 軸: $S$（DM 集中度パラメータ）— 大きいほどブランドロイヤルティが高い

In [ ]:
df_map = df_nbd[["category", "K"]].merge(df_dm[["category", "S"]], on="category")

# カテゴリ名を短縮してラベルを読みやすくする
short_labels = {
    "SOFT DRINKS": "Soft Drinks",
    "FLUID MILK PRODUCTS": "Milk",
    "CHEESE": "Cheese",
    "BAG SNACKS": "Bag Snacks",
    "COLD CEREAL": "Cold Cereal",
    "YOGURT": "Yogurt",
    "FROZEN PIZZA": "Frozen Pizza",
    "SOUP": "Soup",
    "FRZN MEAT/MEAT DINNERS": "Frozen Meat",
    "BAKED BREAD/BUNS/ROLLS": "Bread/Rolls",
}
df_map["label"] = df_map["category"].map(short_labels)

k_med = df_map["K"].median()
s_med = df_map["S"].median()

fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(df_map["K"], df_map["S"], s=80, color="steelblue", zorder=3)

for _, row in df_map.iterrows():
    ax.annotate(
        row["label"],
        xy=(row["K"], row["S"]),
        xytext=(6, 4),
        textcoords="offset points",
        fontsize=10,
    )

# 中央値で象限を区切る参照線
ax.axvline(k_med, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
ax.axhline(s_med, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)

# 象限ラベル
ax.text(df_map["K"].max() * 0.98, df_map["S"].max() * 0.98,
        "ロイヤルティ高\n購入頻度均一", ha="right", va="top", fontsize=9, color="gray")
ax.text(df_map["K"].min() * 1.02, df_map["S"].max() * 0.98,
        "ロイヤルティ高\nヘビー偏在", ha="left", va="top", fontsize=9, color="gray")
ax.text(df_map["K"].max() * 0.98, df_map["S"].min() * 1.02,
        "スイッチング多\n購入頻度均一", ha="right", va="bottom", fontsize=9, color="gray")
ax.text(df_map["K"].min() * 1.02, df_map["S"].min() * 1.02,
        "スイッチング多\nヘビー偏在", ha="left", va="bottom", fontsize=9, color="gray")

ax.set_xlabel("$K$（購入頻度の均一性）", fontsize=12)
ax.set_ylabel("$S$（ブランドロイヤルティ）", fontsize=12)
ax.set_title("K × S カテゴリマッピング", fontsize=14)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../article/fig_ks_map.png", dpi=150, bbox_inches="tight")
plt.show()